In [45]:
import pandas as pd
import requests
import ipaddress
from ipaddress import ip_address
from ipwhois import IPWhois
from ipwhois.exceptions import IPDefinedError, HTTPLookupError
import numpy as np
import os
import re
import time
from pathlib import Path


In [46]:
# 1 - Load input data
csv_file = "cloudtrail.csv"
ct = pd.read_csv(csv_file)
print(f"Shape: {ct.shape}")
print(f"Columns: {list(ct.columns)}")


Shape: (25677, 8)
Columns: ['time', 'source', 'service', 'action', 'agent', 'arn', 'tactic1', 'tactic2']


In [47]:
# 2 - Load Current AWS CIDRs
aws_data = requests.get("https://ip-ranges.amazonaws.com/ip-ranges.json").json()
cidrs = aws_data["prefixes"]  # IPv4 only
aws = pd.DataFrame(cidrs)[["ip_prefix", "region", "service", "network_border_group"]]

# Group by region, service, network_border_group and count
# grouped = aws.groupby(["region", "service", "network_border_group"]).size().reset_index(name="count")
grouped = aws.groupby([ "service"]).size().reset_index(name="count")
grouped_sorted = grouped.sort_values("count", ascending=False)
print(grouped_sorted.to_string(index=False))


                        service  count
                         AMAZON   5776
                            EC2   1767
               ROUTE53_RESOLVER    638
                             S3    418
                    API_GATEWAY    212
                     CLOUDFRONT    203
              GLOBALACCELERATOR    113
          KINESIS_VIDEO_STREAMS    105
                            EBS    100
                   IVS_REALTIME     98
                       DYNAMODB     92
            WORKSPACES_GATEWAYS     70
                      CODEBUILD     60
               MEDIA_PACKAGE_V2     52
                         CLOUD9     50
                 AMAZON_APPFLOW     48
       CLOUDFRONT_ORIGIN_FACING     45
ROUTE53_HEALTHCHECKS_PUBLISHING     39
           EC2_INSTANCE_CONNECT     36
           ROUTE53_HEALTHCHECKS     30
                        ROUTE53     23
           CHIME_VOICECONNECTOR     23
                 AMAZON_CONNECT     21
                    AURORA_DSQL     14
                 CHIME_ME

In [48]:


# 3 - Azure - Stable Microsoft landing page for weekly Service Tags file
landing_url = "https://www.microsoft.com/en-us/download/details.aspx?id=56519"
html = requests.get(landing_url, timeout=30).text

# Find the current ServiceTags_Public_YYYYMMDD.json URL and load Azure CIDRs
m = re.search(r'https://download\.microsoft\.com/[^\"]*ServiceTags_Public_[0-9]{8}\.json', html)
if not m:
    raise RuntimeError("Could not find Azure Service Tags JSON link on Microsoft page.")

json_url = m.group(0)
data = requests.get(json_url, timeout=60).json()

rows = []
for item in data.get("values", []):
    props = item.get("properties", {})
    for prefix in props.get("addressPrefixes", []):
        rows.append({
            "service": item.get("name"),
            "cloud": data.get("cloud"),
            "region": props.get("region"),
            "platform": props.get("platform"),
            "cidr": prefix
        })

azure = pd.DataFrame(rows)
print(f"Loaded {len(azure):,} Azure CIDRs from {json_url}")
azure.head()


Loaded 101,456 Azure CIDRs from https://download.microsoft.com/download/7/1/d/71d86715-5596-4529-9b13-da13a5de5b63/ServiceTags_Public_20260323.json


,service,cloud,region,platform,cidr
0,ActionGroup,Public,,Azure,4.145.74.52/30
1,ActionGroup,Public,,Azure,4.149.254.68/30
2,ActionGroup,Public,,Azure,4.150.239.212/30
3,ActionGroup,Public,,Azure,4.151.103.92/30
4,ActionGroup,Public,,Azure,4.171.31.152/30


In [49]:
# 4 - Normalize shared cloud-range fields and keep provider-specific metadata separate
# Field mappings between AWS and Azure cloud range dataframes
# aws["ip_prefix"]               -> azure["cidr"]
# aws["service"]                 -> azure["service"]
# aws["region"]                  -> azure["region"]
# aws["network_border_group"]    -> no direct Azure equivalent
# no direct AWS equivalent       -> azure["cloud"]
# no direct AWS equivalent       -> azure["platform"]

# Normalized shared fields used in cloud_ranges
# provider                        provider identifier added during normalization
# cidr                            aws["ip_prefix"] / azure["cidr"]
# service                         aws["service"] / azure["service"]
# region                          aws["region"] / azure["region"]
# asn                             not provided in source data; initialized as missing

# Provider-specific fields preserved separately
# network_border_group            AWS only
# cloud                           Azure only
# platform                        Azure only

aws_ranges = aws.assign(
    provider="AWS",
    cidr=aws["ip_prefix"],
    asn=pd.NA,
    cloud=pd.NA,
    platform=pd.NA,
)[["provider", "cidr", "service", "region", "asn", "network_border_group", "cloud", "platform"]]

azure_ranges = azure.assign(
    provider="Azure",
    asn=pd.NA,
    network_border_group=pd.NA,
)[["provider", "cidr", "service", "region", "asn", "network_border_group", "cloud", "platform"]]

cloud_ranges = pd.concat([aws_ranges, azure_ranges], ignore_index=True)
cloud_ranges["asn"] = pd.to_numeric(cloud_ranges["asn"], errors="coerce").astype("Int64")
cloud_ranges.head()


,provider,cidr,service,region,asn,network_border_group,cloud,platform
0,AWS,3.4.12.4/32,AMAZON,eu-west-1,<NA>,eu-west-1,NaN,NaN
1,AWS,3.5.140.0/22,AMAZON,ap-northeast-2,<NA>,ap-northeast-2,NaN,NaN
2,AWS,15.190.244.0/22,AMAZON,ap-east-2,<NA>,ap-east-2,NaN,NaN
3,AWS,15.230.15.29/32,AMAZON,eu-central-1,<NA>,eu-central-1,NaN,NaN
4,AWS,15.230.15.76/31,AMAZON,eu-central-1,<NA>,eu-central-1,NaN,NaN


In [50]:
# Import local mappings
DATA_DIR = Path("lists")
csv_files = sorted(DATA_DIR.glob("*.csv"))

org_to_provider = {
    "AMAZON-02, US": "AWS",
    "AMAZON-AES, US": "AWS",
    "AWS-GOVCLOUD AWS-GOVCLOUD, IE": "AWS",
    "MICROSOFT-CORP-MSN-AS-BLOCK, US": "Azure",
    "JOYENT-INC-, US": "Other",
    "OPENTRANSIT, FR": "Other",
    "SIEMENS-EDC-MIF, US": "Other",
    "ZSCALER-EMEA, CH": "Other",
    "ZSCALER-INC, US": "Other",
    "ZSCALER-SJC1, US": "Other",
}

if not csv_files:
    local = pd.DataFrame(columns=["cidr", "service", "asn", "provider", "region", "source_file"])
    print(f"Warning: no CSV files found in {DATA_DIR.resolve()} — continuing without local mappings")
else:
    local = pd.concat(
        [pd.read_csv(f).assign(source_file=f.name) for f in csv_files],
        ignore_index=True
    )

    if "Unnamed: 0" in local.columns:
        local = local.drop(columns=["Unnamed: 0"])
    elif "" in local.columns:
        local = local.drop(columns=[""])

    local["asn"] = pd.to_numeric(local["asn"], errors="coerce").astype("Int64")
    local["provider"] = local["provider"].map(org_to_provider)

    duplicate_rows = local[local.duplicated(keep=False)].shape[0]
    duplicate_cidrs = local[local.duplicated(subset=["cidr"], keep=False)].shape[0]

    print(f"Loaded {len(csv_files)} file(s), {len(local):,} rows")
    print(f"Exact duplicate rows: {duplicate_rows:,}")
    print(f"Rows sharing the same cidr: {duplicate_cidrs:,}")
    print("Provider counts:")
    print(local["provider"].value_counts(dropna=False).to_string())


Loaded 3 file(s), 769 rows
Exact duplicate rows: 0
Rows sharing the same cidr: 0
Provider counts:
provider
Other    434
NaN      312
AWS       21
Azure      2


In [51]:
cloud_range_base_columns = [
    "provider",
    "cidr",
    "service",
    "region",
    "asn",
    "network_border_group",
    "cloud",
    "platform",
]

df_cloud_ranges = local[["cidr", "service", "asn", "provider", "region"]].copy()
df_cloud_ranges["network_border_group"] = pd.NA
df_cloud_ranges["cloud"] = pd.NA
df_cloud_ranges["platform"] = pd.NA
df_cloud_ranges = df_cloud_ranges[cloud_range_base_columns]

cloud_ranges_base = cloud_ranges[cloud_range_base_columns].copy()


In [52]:
merge_keys = ["cidr", "service", "asn", "provider", "region"]

df_cloud_ranges = df_cloud_ranges.drop_duplicates(subset=merge_keys)

df_cloud_ranges_new = df_cloud_ranges.merge(
    cloud_ranges_base[merge_keys].drop_duplicates(),
    on=merge_keys,
    how="left",
    indicator=True
)
df_cloud_ranges_new = df_cloud_ranges_new[df_cloud_ranges_new["_merge"] == "left_only"].drop(columns=["_merge"])

cloud_ranges_merged = pd.concat(
    [cloud_ranges_base, df_cloud_ranges_new],
    ignore_index=True
)

print(f"Original cloud_ranges rows: {len(cloud_ranges):,}")
print(f"Unique CSV rows considered: {len(df_cloud_ranges):,}")
print(f"CSV rows dropped because cloud_ranges is authoritative: {len(df_cloud_ranges) - len(df_cloud_ranges_new):,}")
print(f"CSV rows appended: {len(df_cloud_ranges_new):,}")
print(f"Merged cloud_ranges rows: {len(cloud_ranges_merged):,}")

cloud_ranges_merged.tail()


Original cloud_ranges rows: 111,491
Unique CSV rows considered: 769
CSV rows dropped because cloud_ranges is authoritative: 0
CSV rows appended: 769
Merged cloud_ranges rows: 112,260


,provider,cidr,service,region,asn,network_border_group,cloud,platform
112255,AWS,170.85.251.0/24,ZSCAL,US,8987,NaN,NaN,NaN
112256,Other,170.85.252.0/24,ZSCALER-MIA4,US,22616,NaN,NaN,NaN
112257,Other,170.85.253.0/24,ZSCALER-MIA4,US,22616,NaN,NaN,NaN
112258,Other,170.85.254.0/24,ZSCAL,US,22616,NaN,NaN,NaN
112259,Other,170.85.255.0/24,ZSCAL,US,22616,NaN,NaN,NaN


In [53]:
# 5 - Read tor map from local file instead of fetching from URL
filename = "map.txt"

try:
    mtime = os.path.getmtime(filename)
    mtime_str = pd.Timestamp(mtime, unit="s", tz="UTC").strftime("%Y-%m-%d %H:%M:%S UTC")
    with open(filename, "r") as f:
        tor_ips = tuple(
            ip_address(line.strip())
            for line in f
            if line.strip()
        )
    print(f"Loaded {len(tor_ips):,} Tor exit node IPs from {filename} (modified {mtime_str})")
except FileNotFoundError:
    tor_ips = tuple()
    print(f"Warning: {filename} not found — Tor exit matching will be skipped")


Loaded 2,009 Tor exit node IPs from map.txt (modified 2025-07-08 19:32:51 UTC)


In [54]:
# 6 - identify public ips vsa. private
def is_public_ip(ip):
    try:
        return ipaddress.ip_address(ip).is_global
    except ValueError:
        return False

# Create a filtered copy
public = ct[ct['source'].apply(is_public_ip)].copy()
print(public.isna().sum())

time       0
source     0
service    0
action     0
agent      0
arn        0
tactic1    0
tactic2    0
dtype: int64


In [55]:
# Convert combined cloud CIDRs to ip_network objects (do this once)
cloud_ranges_merged["network"] = cloud_ranges_merged["cidr"].apply(lambda cidr: ipaddress.ip_network(cidr, strict=False))
cloud_networks = list(cloud_ranges_merged["network"])

# Work with unique public source IPs so repeated CloudTrail rows are labeled once
unique_public_ips = public[["source"]].dropna().drop_duplicates().copy()
unique_public_ips["ip_obj"] = unique_public_ips["source"].apply(ipaddress.ip_address)

def ip_in_cloud(ip_obj, cloud_networks):
    return any(ip_obj in net for net in cloud_networks)

unique_public_ips["in_cloud"] = unique_public_ips["ip_obj"].apply(lambda ip: ip_in_cloud(ip, cloud_networks))
public = public.drop(columns=["in_cloud"], errors="ignore").merge(
    unique_public_ips[["source", "in_cloud"]],
    on="source",
    how="left",
)

match_count = int(public["in_cloud"].fillna(False).sum())
print(f"Rows with source IP in cloud CIDRs: {match_count:,}")


Rows with source IP in cloud CIDRs: 667


In [56]:
# 10 - Label unique public IPs once, then merge the result back onto public
def get_first_cloud_match(ip_obj):
    for row in cloud_ranges_merged.itertuples(index=False):
        if ip_obj in row.network:
            return {
                "provider": row.provider,
                "label": row.service,
                "matched_cidr": row.cidr,
                "matched_region": row.region,
            }
    return {
        "provider": np.nan,
        "label": np.nan,
        "matched_cidr": np.nan,
        "matched_region": np.nan,
    }

# Only run expensive itertuples loop on IPs that passed the fast in_cloud check
cloud_candidates = unique_public_ips[unique_public_ips["in_cloud"]].copy()

ip_labels = pd.concat(
    [cloud_candidates, cloud_candidates["ip_obj"].apply(get_first_cloud_match).apply(pd.Series)],
    axis=1,
)

public = public.drop(columns=["provider", "label", "matched_cidr", "matched_region"], errors="ignore").merge(
    ip_labels[["source", "provider", "label", "matched_cidr", "matched_region"]],
    on="source",
    how="left",
)

provider_counts = public["provider"].value_counts(dropna=True)
print("Labeled public rows by provider:")
print(provider_counts.reindex(["AWS", "Azure"], fill_value=0).to_string())
print()
print(public[["source", "provider", "label", "matched_cidr"]].dropna(subset=["label"]).head())


Labeled public rows by provider:
provider
AWS      667
Azure      0

           source provider   label    matched_cidr
1    72.21.217.72      AWS  AMAZON  72.21.192.0/19
2   72.21.217.154      AWS  AMAZON  72.21.192.0/19
3    72.21.217.54      AWS  AMAZON  72.21.192.0/19
14   72.21.217.75      AWS  AMAZON  72.21.192.0/19
15   72.21.217.49      AWS  AMAZON  72.21.192.0/19


In [57]:
# Tor exit node matching
tor_ip_set = set(tor_ips)

def is_tor_exit(ip_str):
    try:
        return ipaddress.ip_address(ip_str) in tor_ip_set
    except ValueError:
        return False

public["tor_match"] = public["source"].apply(is_tor_exit)
public.loc[public["tor_match"], "label"] = "tor"

tor_matches_df = public[public["tor_match"]].copy()
print("Source IPs found in Tor exit list:")
print(tor_matches_df["source"].drop_duplicates().to_string(index=False))


Source IPs found in Tor exit list:
100.42.28.64


In [58]:
# Count unique combinations of provider and label
public.groupby(["provider", "label"], dropna=False).size().reset_index(name="count").sort_values("count", ascending=False)

,provider,label,count
0,AWS,AMAZON,667
2,NaN,NaN,36
1,NaN,tor,1


In [ ]:
# Filter to unlabeled public IPs
filtered = public[public["label"].isna()]
unique_ips = filtered["source"].dropna().unique()
print(f"Unlabeled public IPs: {len(unique_ips)}")
print(unique_ips) 

Unlabeled public IPs: 1
['223.0.113.35']


In [60]:

# 18 - Use the unlabeled public IPs gathered above
publicIps = unique_ips

def _first_contact_value(contact, field):
    values = contact.get(field) or []
    if not values:
        return None
    first = values[0]
    if isinstance(first, dict):
        return first.get("value")
    return first

def lookup_cidr(ip):
    try:
        ipaddress.ip_address(ip)
        result = IPWhois(ip).lookup_rdap(depth=1)
        net = result.get("network") or {}
        cidr = net.get("cidr")
        if not cidr:
            start = net.get("start_address")
            end = net.get("end_address")
            cidr = f"{start}-{end}" if start and end else None

        objects = result.get("objects") or {}
        registrant = None
        abuse = None
        for obj in objects.values():
            roles = obj.get("roles") or []
            if registrant is None and "registrant" in roles:
                registrant = obj
            if abuse is None and "abuse" in roles:
                abuse = obj

        registrant_contact = (registrant or {}).get("contact") or {}
        abuse_contact = (abuse or {}).get("contact") or {}
        network_events = net.get("events") or []
        registered = next((e.get("timestamp") for e in network_events if e.get("action") == "registration"), None)
        last_changed = next((e.get("timestamp") for e in network_events if e.get("action") == "last changed"), None)

        return {
            "ip": ip,
            "cidr": cidr,
            "asn": result.get("asn") or None,
            "asn_cidr": result.get("asn_cidr") or None,
            "asn_registry": result.get("asn_registry") or None,
            "asn_country": result.get("asn_country_code") or None,
            "asn_description": result.get("asn_description") or None,
            "org": (net.get("name") or result.get("asn_description") or "").strip() or None,
            "network_country": net.get("country") or None,
            "country": result.get("asn_country_code") or net.get("country") or None,
            "network_handle": net.get("handle") or None,
            "parent_handle": net.get("parent_handle") or None,
            "network_type": net.get("type") or None,
            "ip_version": net.get("ip_version") or None,
            "status": ", ".join(net.get("status") or []) or None,
            "start_address": net.get("start_address") or None,
            "end_address": net.get("end_address") or None,
            "registered": registered,
            "last_changed": last_changed,
            #"registrant_name": registrant_contact.get("name") or None,
            #"registrant_kind": registrant_contact.get("kind") or None,
            #"registrant_email": _first_contact_value(registrant_contact, "email"),
            #"registrant_phone": _first_contact_value(registrant_contact, "phone"),
            #"abuse_name": abuse_contact.get("name") or None,
            #"abuse_email": _first_contact_value(abuse_contact, "email"),
            #"abuse_phone": _first_contact_value(abuse_contact, "phone"),
            "entities": ", ".join(result.get("entities") or []) or None,
            "error": None,
        }
    except Exception as e:
        return {
            "ip": ip,
            "cidr": None,
            "asn": None,
            "asn_cidr": None,
            "asn_registry": None,
            "asn_country": None,
            "asn_description": None,
            "org": None,
            "network_country": None,
            "country": None,
            "network_handle": None,
            "parent_handle": None,
            "network_type": None,
            "ip_version": None,
            "status": None,
            "start_address": None,
            "end_address": None,
            "registered": None,
            "last_changed": None,
            #"registrant_name": None,
            #"registrant_kind": None,
            #"registrant_email": None,
            #"registrant_phone": None,
            #"abuse_name": None,
            #"abuse_email": None,
            #"abuse_phone": None,
            "entities": None,
            "error": str(e),
        }

records = []
seen = set()
for ip in publicIps:
    if ip in seen:
        continue
    seen.add(ip)
    records.append(lookup_cidr(ip))
    time.sleep(0.2)  # polite spacing for RDAP servers

cidrFrame = pd.DataFrame(records)
cidrFrame


,ip,cidr,asn,asn_cidr,asn_registry,asn_country,asn_description,org,network_country,country,...,parent_handle,network_type,ip_version,status,start_address,end_address,registered,last_changed,entities,error
0,223.0.113.35,223.0.0.0/15,NA,NA,apnic,CN,NA,CNBIDCC,CN,CN,...,None,ALLOCATED PORTABLE,v4,active,223.0.0.0,223.1.255.255,2011-03-31T03:02:22Z,2021-11-02T07:48:00Z,"IRT-CNBIDCC-CN, YW7200-AP",None


In [61]:
# Merge RDAP results back onto public
rdap_labels = cidrFrame.rename(columns={"ip": "source"})

# A row is useful if at least one of the key fields is not NaN
useful_fields = ["org", "entities", "country"]
rdap_useful = rdap_labels.dropna(subset=useful_fields, how="all")
rdap_skipped = rdap_labels[rdap_labels[useful_fields].isna().all(axis=1)]

if not rdap_skipped.empty:
    print("Skipped (no useful RDAP data):")
    print(rdap_skipped["source"].to_string(index=False))

for _, row in rdap_useful.iterrows():
    mask = public["source"] == row["source"]
    if pd.notna(row["org"]):
        public.loc[mask, "provider"] = row["org"]
    if pd.notna(row["entities"]):
        public.loc[mask, "label"] = row["entities"]
    if pd.notna(row["country"]):
        public.loc[mask, "matched_region"] = row["country"]

print(f"Rows labeled from RDAP: {len(rdap_useful):,}")
print(f"Rows skipped (no data): {len(rdap_skipped):,}")


,time,source,service,action,agent,arn,tactic1,tactic2,in_cloud,provider,...,parent_handle,network_type,ip_version,status,start_address,end_address,registered,last_changed,entities,error
0,2018-09-30T17:30:21.000Z,223.0.113.35,signin.amazonaws.com,ConsoleLogin,AWSBFF BFFiOS/1.19.6 (1015) Mobile iOS/11.4.1 ...,arn:aws:iam::123456789012:root,Persistence,Initial Access,False,NaN,...,None,ALLOCATED PORTABLE,v4,active,223.0.0.0,223.1.255.255,2011-03-31T03:02:22Z,2021-11-02T07:48:00Z,"IRT-CNBIDCC-CN, YW7200-AP",None
1,2018-09-30T17:30:30.000Z,223.0.113.35,ec2.amazonaws.com,DescribeAddresses,mobileconsole.amazonaws.com,arn:aws:iam::123456789012:root,Discovery,-,False,NaN,...,None,ALLOCATED PORTABLE,v4,active,223.0.0.0,223.1.255.255,2011-03-31T03:02:22Z,2021-11-02T07:48:00Z,"IRT-CNBIDCC-CN, YW7200-AP",None
2,2018-09-30T17:30:34.000Z,223.0.113.35,ec2.amazonaws.com,DescribeAddresses,mobileconsole.amazonaws.com,arn:aws:iam::123456789012:root,Discovery,-,False,NaN,...,None,ALLOCATED PORTABLE,v4,active,223.0.0.0,223.1.255.255,2011-03-31T03:02:22Z,2021-11-02T07:48:00Z,"IRT-CNBIDCC-CN, YW7200-AP",None
